# SQL for Data Platforms
## Part 11: Procedural SQL — Stored Procedures, Triggers & PL/SQL

Everything through Part 10 was **declarative** SQL — you describe the result, the engine figures
out how to get it. **Procedural SQL** adds variables, control flow, and stored, reusable
routines — a different tool, still SQL. This is the one module in the series that needs a real
server: neither SQLite nor DuckDB implement real stored procedures/triggers, so its one runnable
demo uses the **Docker Postgres** container flagged as optional back in Part 10.

**Accuracy note:** the Oracle **PL/SQL** and SQL Server/Fabric **T-SQL** sections below are
reference-level — capability descriptions, not commands run against a real Oracle/SQL Server
instance from this repo. Verify exact syntax against current vendor docs before relying on it.

## Section 1 — What procedural SQL adds

- **Variables** — hold an intermediate value across statements (`DECLARE`).
- **Control flow** — `IF`/`ELSE`, loops (`LOOP`/`WHILE`/`FOR`) — plain SQL has no loops.
- **Stored procedures/functions** — a named, saved, parameterized routine, callable repeatedly.
- **Triggers** — a routine that fires automatically on `INSERT`/`UPDATE`/`DELETE` against a table.
- **Cursors** — iterate row-by-row over a result set when set-based SQL genuinely can't express
  the logic (rare — reach for this last, not first).
- **Exception handling** — catch and react to errors inside the routine itself.

## Section 2 — Runnable demo: PL/pgSQL via Docker Postgres

**Requires the optional container from Part 10:**
```bash
docker compose -f docker/docker-compose.yml up -d
```
If it isn't running, this section's cells will raise a connection error — that's expected; every
other module in this repo runs with zero setup, this is the one exception.

In [1]:
import psycopg2

try:
    pg = psycopg2.connect(host="localhost", port=5432, dbname="sqlfordata", user="postgres", password="postgres")
    pg.autocommit = True
    cur = pg.cursor()
    print("connected to Docker Postgres")
except Exception as e:
    pg = None
    print(f"Docker Postgres not running (this is the one optional module) -- {e}")
    print("Start it with: docker compose -f docker/docker-compose.yml up -d")

Docker Postgres not running (this is the one optional module) -- connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?

Start it with: docker compose -f docker/docker-compose.yml up -d


In [2]:
if pg:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id INTEGER PRIMARY KEY,
            name TEXT NOT NULL,
            price NUMERIC(10,2) NOT NULL,
            stock_quantity INTEGER NOT NULL
        )
    """)
    cur.execute("TRUNCATE products")
    cur.execute("""
        INSERT INTO products VALUES
            (1, 'Wireless Headphones', 129.99, 45),
            (2, 'Laptop Stand', 49.99, 3)
    """)
    print("products table ready")

### A stored function with control flow

Reorder a product automatically when its stock drops below a threshold — logic a plain `UPDATE`
can't express (branching on a condition, returning a status message).

In [3]:
if pg:
    cur.execute("""
        CREATE OR REPLACE FUNCTION restock_if_low(p_id INTEGER, p_threshold INTEGER, p_restock_qty INTEGER)
        RETURNS TEXT AS $$
        DECLARE
            current_stock INTEGER;
        BEGIN
            SELECT stock_quantity INTO current_stock FROM products WHERE id = p_id;

            IF current_stock IS NULL THEN
                RETURN 'Product not found';
            ELSIF current_stock < p_threshold THEN
                UPDATE products SET stock_quantity = stock_quantity + p_restock_qty WHERE id = p_id;
                RETURN 'Restocked: ' || current_stock || ' -> ' || (current_stock + p_restock_qty);
            ELSE
                RETURN 'Stock sufficient, no action taken';
            END IF;
        END;
        $$ LANGUAGE plpgsql;
    """)

    cur.execute("SELECT restock_if_low(2, 10, 50)")   # Laptop Stand: stock=3, below threshold=10
    print(cur.fetchone()[0])
    cur.execute("SELECT restock_if_low(1, 10, 50)")   # Headphones: stock=45, above threshold
    print(cur.fetchone()[0])

### A trigger — logic that fires automatically

Log every price change to an audit table, without any application code having to remember to call
it.

In [4]:
if pg:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS price_audit (
            product_id INTEGER, old_price NUMERIC(10,2), new_price NUMERIC(10,2), changed_at TIMESTAMP
        )
    """)
    cur.execute("TRUNCATE price_audit")
    cur.execute("""
        CREATE OR REPLACE FUNCTION log_price_change() RETURNS TRIGGER AS $$
        BEGIN
            IF NEW.price != OLD.price THEN
                INSERT INTO price_audit VALUES (OLD.id, OLD.price, NEW.price, now());
            END IF;
            RETURN NEW;
        END;
        $$ LANGUAGE plpgsql;
    """)
    cur.execute("DROP TRIGGER IF EXISTS trg_price_change ON products")
    cur.execute("""
        CREATE TRIGGER trg_price_change
        BEFORE UPDATE ON products
        FOR EACH ROW EXECUTE FUNCTION log_price_change();
    """)

    cur.execute("UPDATE products SET price = 139.99 WHERE id = 1")   # fires the trigger
    cur.execute("SELECT * FROM price_audit")
    print(cur.fetchall())

In [5]:
if pg:
    cur.close()
    pg.close()
    print("connection closed")

## Section 3 — Reference: Oracle PL/SQL

*(Not executed here — capability description only; verify exact syntax against current Oracle
docs before relying on it.)*

PL/SQL wraps SQL in `BEGIN`/`END` blocks with the same DECLARE/control-flow shape as PL/pgSQL.
Distinctive Oracle features: **`%ROWTYPE`** (a variable that mirrors a table row's structure
without redeclaring every column), **`%TYPE`** (mirrors a single column's type), and **packages**
(`PACKAGE`/`PACKAGE BODY` — a named, versioned bundle of related procedures/functions, closer to a
module than a single stored routine):

```sql
CREATE OR REPLACE PROCEDURE restock_if_low(p_id IN products.id%TYPE, p_threshold IN NUMBER) IS
    v_product products%ROWTYPE;
BEGIN
    SELECT * INTO v_product FROM products WHERE id = p_id;
    IF v_product.stock_quantity < p_threshold THEN
        UPDATE products SET stock_quantity = stock_quantity + 50 WHERE id = p_id;
    END IF;
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        DBMS_OUTPUT.PUT_LINE('Product not found');
END;
/
```

## Section 4 — Reference: SQL Server / Microsoft Fabric T-SQL

*(Not executed here — capability description only; verify exact syntax against current Microsoft
docs before relying on it.)*

T-SQL uses `BEGIN`/`END` blocks too, but structured error handling is **`TRY`/`CATCH`** (closer to
mainstream language exception handling than PL/SQL's `EXCEPTION WHEN`), and **table variables**
(`DECLARE @t TABLE (...)`) hold a result set in memory for the duration of a batch:

```sql
CREATE OR ALTER PROCEDURE RestockIfLow @ProductId INT, @Threshold INT AS
BEGIN
    BEGIN TRY
        DECLARE @CurrentStock INT = (SELECT stock_quantity FROM products WHERE id = @ProductId);
        IF @CurrentStock < @Threshold
            UPDATE products SET stock_quantity = stock_quantity + 50 WHERE id = @ProductId;
    END TRY
    BEGIN CATCH
        PRINT ERROR_MESSAGE();
    END CATCH
END;
```

Microsoft Fabric's Warehouse item supports a T-SQL subset for querying/procedures; verify current
feature coverage against Fabric docs, since it does not yet match on-prem SQL Server 1:1.

## Section 5 — Do cloud warehouses still need this?

Snowflake and BigQuery both answer with their own **scripting** extensions (Snowflake Scripting,
BigQuery scripting `BEGIN...END` blocks with variables/loops) rather than adopting PL/SQL or T-SQL
directly — same procedural ideas, different platform-native syntax (see Part 12). The dbt/ELT
culture this whole series otherwise follows pushes logic into **SQL models + Python**, not stored
procedures — reach for procedural SQL when the logic genuinely needs to live *inside* the database
(a trigger reacting to a write, in particular), not as a default way to write transformations.

## Best Practices — Procedural SQL

- Prefer set-based SQL (a single `UPDATE`/`MERGE`) over a cursor loop wherever the same result can
  be expressed set-based — row-by-row procedural logic is almost always slower.
- Triggers are powerful and easy to lose track of — they run implicitly, with no caller visible in
  the code that changed the row. Document every trigger's existence somewhere a future reader will
  actually find it.
- Keep stored procedures thin where possible; version-controlled SQL models (Part 10's dbt project)
  are easier to test, review, and diff than logic embedded in the database itself.

## Next

**Part 12 — Same Query, Every Platform** is where the platform-specific syntax promised
throughout this series — BigQuery, Snowflake, Databricks SQL, Fabric, Redshift — all lives.